In [ ]:
!pip install -q datasets
!pip install krippendorff

In [ ]:
!pip install transformers

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:

import os
os.chdir("/content/gdrive/MyDrive/LLM/CBLLMMODEL")


# Prepare DataSets

In [ ]:
import pandas as pd

df_majority = pd.read_excel("TCMB_ALL_MODELS_MAJORITY.xlsx", index_col="No")
neutral_df = df_majority[df_majority["majority_vote"] == "neutral"].sample(n=2400, random_state=42)
hawkish_df = df_majority[df_majority["majority_vote"] == "hawkish"].sample(n=2400, random_state=42)
dovish_df  = df_majority[df_majority["majority_vote"] == "dovish"].sample(n=2400, random_state=42)
balanced_df = pd.concat([neutral_df, hawkish_df, dovish_df]).sample(frac=1, random_state=42).reset_index(drop=True)
label_map = {"neutral": 0, "hawkish": 1, "dovish": 2}
balanced_df["label"] = balanced_df["majority_vote"].map(label_map)
final_df = balanced_df[["sentence", "label"]].rename(columns={"sentence": "text"})
final_df.to_csv("BALANCED_DATASET.csv", index=False)
final_df.head(3)


In [ ]:
print(len(df_majority))

In [ ]:
import pandas as pd
import krippendorff

# Örnek olarak CSV/Excel yüklenebilir
# df = pd.read_excel("veri.xlsx")

# Sadece ilgili sütunları al
ratings = df_majority[["DeepSeek","GPT", "GEMINI", "majority_vote"]]

# Krippendorff paketi string label’larla çalışmaz, sayısal hale getirmeliyiz
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
ratings_encoded = ratings.apply(encoder.fit_transform)

# Transpose etmek gerekiyor çünkü krippendorff modülü gözlem bazlı (columns = annotators)
ratings_matrix = ratings_encoded.T.values

# Alpha'yı hesapla (nominal için)
alpha = krippendorff.alpha(reliability_data=ratings_matrix, level_of_measurement='nominal')
print("Krippendorff’s Alpha:", round(alpha, 4))

In [ ]:
# Her model için etiket sayılarını hesapla
gpt_counts = df_majority["GPT"].value_counts()
gemini_counts = df_majority["GEMINI"].value_counts()
deepseek_counts = df_majority["DeepSeek"].value_counts()
majority_counts = df_majority["majority_vote"].value_counts()

# Üçünü birleştir ve eksik değerleri 0 ile doldur
result = pd.DataFrame({
    "GPT": gpt_counts,
    "GEMINI": gemini_counts,
    "DeepSeek": deepseek_counts,
    "Majority": majority_counts
}).fillna(0).astype(int)

# Sonuçları göster
print(result)

In [ ]:
filtered_df = df_majority[(df_majority["GPT"] == df_majority["GEMINI"]) & (df_majority["GPT"] == df_majority["DeepSeek"])]

# Her model için etiket sayılarını hesapla
gpt_counts = filtered_df["GPT"].value_counts()
gemini_counts = filtered_df["GEMINI"].value_counts()
deepseek_counts = filtered_df["DeepSeek"].value_counts()

# Üçünü birleştir ve eksik değerleri 0 ile doldur
result = pd.DataFrame({
    "GPT": gpt_counts,
    "GEMINI": gemini_counts,
    "DeepSeek": deepseek_counts
}).fillna(0).astype(int)

# Sonuçları göster
print(result)



In [ ]:
filtered_df_2 = df_majority[(df_majority["GPT"] != df_majority["GEMINI"]) &
                            (df_majority["GPT"] != df_majority["DeepSeek"]) &
                            (df_majority["GEMINI"] == df_majority["DeepSeek"])]
filtered_df_2.head()

In [ ]:
len(filtered_df_2)

In [ ]:
filtered_df["GPT"].value_counts()

In [ ]:
filtered_df["GEMINI"].value_counts()

In [ ]:
filtered_df["DeepSeek"].value_counts()

# ROBERTA

In [ ]:
# 🔹 CSV'den veri yükle (text, label sütunları olmalı)
df = pd.read_csv("BALANCED_DATASET.csv")

# 🔹 label: neutral=0, hawkish=1, dovish=2
label_map = {0: "neutral", 1: "hawkish", 2: "dovish"}
id2label = {v: k for k, v in label_map.items()}

# 🔹 Hugging Face Dataset'e çevir
dataset = Dataset.from_pandas(df)

# 🔹 Train-test split
dataset = dataset.train_test_split(test_size=0.2)

# 🔹 Tokenizer & Model
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# 🔹 Tokenize
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

tokenized = dataset.map(tokenize_fn, batched=True)

# 🔹 Değerlendirme Fonksiyonu
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# 🔹 Eğitim ayarları
training_args = TrainingArguments(
    output_dir="./roberta_hawkish_dovish",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    num_train_epochs=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none"
)

# 🔹 Trainer objesi
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# 🔹 Eğitimi başlat
trainer.train()

# 🔹 Modeli ve tokenizer'ı kaydet
trainer.save_model("./roberta_hawkish_dovish_model")
tokenizer.save_pretrained("./roberta_hawkish_dovish_model")

In [ ]:

from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

preds = trainer.predict(tokenized["test"])
y_true = tokenized["test"]["label"]
y_pred = np.argmax(preds.predictions, axis=1)
print(classification_report(y_true, y_pred, target_names=["neutral", "hawkish", "dovish"]))
y_true = tokenized["test"]["label"]
y_pred = np.argmax(preds.predictions, axis=1)
acc = accuracy_score(y_true, y_pred)
print(f"✅ Accuracy: {acc:.4f}")

In [ ]:

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np
# Model klasöründen yükle
model_path = "./roberta_hawkish_dovish_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

In [ ]:
id2label = {0: "neutral", 1: "hawkish", 2: "dovish"}

In [ ]:
def predict_sentence(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.nn.functional.softmax(logits, dim=-1)
        predicted_class = torch.argmax(probs, dim=1).item()
        return {
            "text": text,
            "prediction": id2label[predicted_class],
            "probabilities": dict(zip(id2label.values(), probs[0].tolist()))
        }


In [ ]:
example = """The Monetary Policy Committee has concluded that the data announced
in the recent period since the last meeting did not lead to a significant change
in the inflation and monetary policy outlook.
"""

result = predict_sentence(example)
print("Metin:", result["text"])
print("Tahmin:", result["prediction"])
print("Olasılıklar:", result["probabilities"])

In [ ]:
from transformers import pipeline

# Load the classifier pipeline
classifier = pipeline("text-classification", model="mrince/CBRT-RoBERTa-HawkishDovish-Classifier")

# Example sentence from a monetary policy context
sentence = "On the other hand, the recent deceleration in economic activity may curb services inflation."

# Perform classification
result = classifier(sentence)
print(result)

# Output example:
# [{'label': 'hawkish', 'score': 0.91}]

# Label meanings:
# - 'hawkish [Label_1]': Tightening bias, upward rate signal. Indicates prioritization of inflation control.
# - 'dovish [Label_2]' : Easing bias, downward/inflation-tolerant tone. Indicates support for growth/stimulus.
# - 'neutral [Label_2]': Informational or balanced tone without a clear policy stance.


In [ ]:
# Example sentence from a monetary policy context
sentence = "On the other hand, the recent deceleration in economic activity may curb services inflation.."

# Perform classification
result = classifier(sentence)
print(result[0]["label"])

# ROBERTA LARGE

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# 🔹 CSV'den veri yükle
df = pd.read_csv("BALANCED_DATASET.csv")

# 🔹 label mapping
label_map = {0: "neutral", 1: "hawkish", 2: "dovish"}
id2label = {v: k for k, v in label_map.items()}

# 🔹 Hugging Face Dataset'e çevir
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

# 🔹 Tokenizer & Model (GÜNCELLENEN KISIM)
model_name = "roberta-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# 🔹 Tokenization
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

tokenized = dataset.map(tokenize_fn, batched=True)

# 🔹 Değerlendirme metrikleri
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# 🔹 Eğitim ayarları (BATCH 8 olarak düşürüldü)
training_args = TrainingArguments(
    output_dir="./roberta_large_hawkish_dovish",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    num_train_epochs=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none"
)



# 🔹 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# 🔹 Eğitimi başlat
trainer.train()

# 🔹 Modeli kaydet
trainer.save_model("./roberta_large_hawkish_dovish_model")
tokenizer.save_pretrained("./roberta_large_hawkish_dovish_model")

In [ ]:

from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

preds = trainer.predict(tokenized["test"])
y_true = tokenized["test"]["label"]
y_pred = np.argmax(preds.predictions, axis=1)
print(classification_report(y_true, y_pred, target_names=["neutral", "hawkish", "dovish"]))
y_true = tokenized["test"]["label"]
y_pred = np.argmax(preds.predictions, axis=1)
acc = accuracy_score(y_true, y_pred)
print(f"✅ Accuracy: {acc:.4f}")


# DİSTİLBERT

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report

# 🔹 CSV'den veri yükle
df = pd.read_csv("BALANCED_DATASET.csv")

# 🔹 label mapping
label_map = {0: "neutral", 1: "hawkish", 2: "dovish"}
id2label = {v: k for k, v in label_map.items()}

# 🔹 HF Dataset'e çevir ve train-test ayır
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

# 🔹 Model & Tokenizer (DISTILBERT kullanılıyor)
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# 🔹 Tokenization
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

tokenized = dataset.map(tokenize_fn, batched=True)

# 🔹 Değerlendirme fonksiyonu
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# 🔹 Eğitim ayarları
training_args = TrainingArguments(
    output_dir="./distilbert_hawkish_dovish",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    num_train_epochs=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none"
)

# 🔹 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# 🔹 Eğitimi başlat
trainer.train()

# 🔹 Modeli ve tokenizer'ı kaydet
trainer.save_model("./distilbert_hawkish_dovish_model")
tokenizer.save_pretrained("./distilbert_hawkish_dovish_model")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

preds = trainer.predict(tokenized["test"])
y_true = tokenized["test"]["label"]
y_pred = np.argmax(preds.predictions, axis=1)
print(classification_report(y_true, y_pred, target_names=["neutral", "hawkish", "dovish"]))
y_true = tokenized["test"]["label"]
y_pred = np.argmax(preds.predictions, axis=1)
acc = accuracy_score(y_true, y_pred)
print(f"✅ Accuracy: {acc:.4f}")

# DEBERTA V3

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# 🔹 CSV'den veri yükle
df = pd.read_csv("BALANCED_DATASET.csv")

# 🔹 label mapping
label_map = {0: "neutral", 1: "hawkish", 2: "dovish"}
id2label = {v: k for k, v in label_map.items()}

# 🔹 HF Dataset'e çevir
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

# 🔹 DeBERTa Tokenizer & Model
model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# 🔹 Tokenization
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

tokenized = dataset.map(tokenize_fn, batched=True)

# 🔹 Değerlendirme fonksiyonu
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# 🔹 Eğitim ayarları
training_args = TrainingArguments(
    output_dir="./deberta_hawkish_dovish",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    num_train_epochs=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none"
)

# 🔹 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# 🔹 Eğitimi başlat
trainer.train()

# 🔹 Modeli kaydet
trainer.save_model("./deberta_hawkish_dovish_model")
tokenizer.save_pretrained("./deberta_hawkish_dovish_model")


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

preds = trainer.predict(tokenized["test"])
y_true = tokenized["test"]["label"]
y_pred = np.argmax(preds.predictions, axis=1)
print(classification_report(y_true, y_pred, target_names=["neutral", "hawkish", "dovish"]))
y_true = tokenized["test"]["label"]
y_pred = np.argmax(preds.predictions, axis=1)
acc = accuracy_score(y_true, y_pred)
print(f"✅ Accuracy: {acc:.4f}")

# Electra

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# 🔹 1. Veri Yükleme
df = pd.read_csv("BALANCED_DATASET.csv")

# 🔹 2. Label Mapping
label_map = {0: "neutral", 1: "hawkish", 2: "dovish"}
id2label = {v: k for k, v in label_map.items()}

# 🔹 3. HF Dataset ve Train-Test Ayırma
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

# 🔹 4. ELECTRA Tokenizer & Model
model_name = "google/electra-base-discriminator"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# 🔹 5. Tokenization
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

tokenized = dataset.map(tokenize_fn, batched=True)

# 🔹 6. Değerlendirme Fonksiyonu
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# 🔹 7. Eğitim Ayarları
training_args = TrainingArguments(
    output_dir="./electra_hawkish_dovish_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    num_train_epochs=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none"
)

# 🔹 8. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# 🔹 9. Eğitimi Başlat
trainer.train()

# 🔹 10. Modeli Kaydet
trainer.save_model("./electra_hawkish_dovish_model")
tokenizer.save_pretrained("./electra_hawkish_dovish_model")


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

preds = trainer.predict(tokenized["test"])
y_true = tokenized["test"]["label"]
y_pred = np.argmax(preds.predictions, axis=1)
print(classification_report(y_true, y_pred, target_names=["neutral", "hawkish", "dovish"]))
y_true = tokenized["test"]["label"]
y_pred = np.argmax(preds.predictions, axis=1)
acc = accuracy_score(y_true, y_pred)
print(f"✅ Accuracy: {acc:.4f}")

# ALBERT

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# 🔹 1. Veri Yükleme
df = pd.read_csv("BALANCED_DATASET.csv")

# 🔹 2. Label Mapping
label_map = {0: "neutral", 1: "hawkish", 2: "dovish"}
id2label = {v: k for k, v in label_map.items()}

# 🔹 3. Hugging Face Dataset'e Dönüştür
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

# 🔹 4. Tokenizer ve Model (ALBERT)
model_name = "albert-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# 🔹 5. Tokenization
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

tokenized = dataset.map(tokenize_fn, batched=True)

# 🔹 6. Değerlendirme Fonksiyonu
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# 🔹 7. Eğitim Ayarları
training_args = TrainingArguments(
    output_dir="./albert_hawkish_dovish_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    num_train_epochs=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none"
)

# 🔹 8. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# 🔹 9. Eğitimi Başlat
trainer.train()

# 🔹 10. Modeli Kaydet
trainer.save_model("./albert_hawkish_dovish_model")
tokenizer.save_pretrained("./albert_hawkish_dovish_model")


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

preds = trainer.predict(tokenized["test"])
y_true = tokenized["test"]["label"]
y_pred = np.argmax(preds.predictions, axis=1)
print(classification_report(y_true, y_pred, target_names=["neutral", "hawkish", "dovish"]))
y_true = tokenized["test"]["label"]
y_pred = np.argmax(preds.predictions, axis=1)
acc = accuracy_score(y_true, y_pred)
print(f"✅ Accuracy: {acc:.4f}")

# LONGFORMER

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# 🔹 1. CSV'den veri yükle
df = pd.read_csv("BALANCED_DATASET.csv")

# 🔹 2. Label mapping
label_map = {0: "neutral", 1: "hawkish", 2: "dovish"}
id2label = {v: k for k, v in label_map.items()}

# 🔹 3. Hugging Face Dataset'e dönüştür
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

# 🔹 4. Tokenizer & Model (Longformer)
model_name = "allenai/longformer-base-4096"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# 🔹 5. Tokenization (uzun girişlere uygun padding ve truncation ayarı!)
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=4096)

tokenized = dataset.map(tokenize_fn, batched=True)

# 🔹 6. Değerlendirme metrikleri
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# 🔹 7. Eğitim ayarları
training_args = TrainingArguments(
    output_dir="./longformer_hawkish_dovish_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=4,  # VRAM için düşürüldü
    num_train_epochs=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none"
)

# 🔹 8. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# # 🔹 9. Eğitimi başlat
# trainer.train()

# # 🔹 10. Modeli kaydet
# trainer.save_model("./longformer_hawkish_dovish_model")
# tokenizer.save_pretrained("./longformer_hawkish_dovish_model")


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

preds = trainer.predict(tokenized["test"])
y_true = tokenized["test"]["label"]
y_pred = np.argmax(preds.predictions, axis=1)
print(classification_report(y_true, y_pred, target_names=["neutral", "hawkish", "dovish"]))
y_true = tokenized["test"]["label"]
y_pred = np.argmax(preds.predictions, axis=1)
acc = accuracy_score(y_true, y_pred)
print(f"✅ Accuracy: {acc:.4f}")

# MACHINE LEARNING MODELS

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
import pandas as pd

In [ ]:
# CSV dosyasını yükle
df = pd.read_csv("BALANCED_DATASET.csv")

# Veri ve etiket
X = df["text"]
y = df["label"]  # Burada 0: neutral, 1: hawkish, 2: dovish

# Eğitim ve test bölmesi
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# TF-IDF vektörleştirici
vectorizer = TfidfVectorizer(max_features=5000)  # daha düşük boyutlu vektörlerle başlamak için

# Veriye uygula
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


## RANDOM FOREST

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_vec, y_train)
y_pred_rf = rf.predict(X_test_vec)

print("🔍 Random Forest Classification Report")
print(classification_report(y_test, y_pred_rf, target_names=["neutral", "hawkish", "dovish"]))


## SUPPORT VECTOR MACHINE

In [ ]:
svm = LinearSVC()
svm.fit(X_train_vec, y_train)
y_pred_svm = svm.predict(X_test_vec)

print("🔍 Support Vector Machine (LinearSVC) Classification Report")
print(classification_report(y_test, y_pred_svm, target_names=["neutral", "hawkish", "dovish"]))


## DECISION TREE

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_vec, y_train)
y_pred_dt = dt.predict(X_test_vec)

print("🔍 Decision Tree Classification Report")
print(classification_report(y_test, y_pred_dt, target_names=["neutral", "hawkish", "dovish"]))


## Multinomial Naive Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train_vec, y_train)
y_pred_nb = nb.predict(X_test_vec)

print("🔍 Multinomial Naive Bayes Classification Report")
print(classification_report(y_test, y_pred_nb, target_names=["neutral", "hawkish", "dovish"]))


## Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_vec, y_train)
y_pred_lr = lr.predict(X_test_vec)

print("🔍 Logistic Regression Classification Report")
print(classification_report(y_test, y_pred_lr, target_names=["neutral", "hawkish", "dovish"]))


## Gradient Boosting

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(use_label_encoder=False, eval_metric="mlogloss")
xgb.fit(X_train_vec, y_train)
y_pred_xgb = xgb.predict(X_test_vec)

print("🔍 XGBoost Classification Report")
print(classification_report(y_test, y_pred_xgb, target_names=["neutral", "hawkish", "dovish"]))


## K-Nearest Neighbors (KNN)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_vec, y_train)
y_pred_knn = knn.predict(X_test_vec)

print("🔍 K-Nearest Neighbors Classification Report")
print(classification_report(y_test, y_pred_knn, target_names=["neutral", "hawkish", "dovish"]))
